# Stage 1 — Baseline ensemble: CatBoost, LightGBM, logistic regression

**Question:** can a clean, leak-free ensemble beat the earlier best of 0.10926 (single CatBoost + Optuna)?

- Repeated stratified 5-fold CV with identical folds for every model; out-of-fold (OOF) predictions for blending.
- CatBoost (GPU), LightGBM with native categoricals, and logistic regression on one-hot pairwise feature crosses.
- Log-loss-optimal probability blend; the 0.5 cutoff only moves if the gain beats seed-to-seed noise.

| result | CV logloss (blend) | public LB | wrong rows / 10,086 |
|---|---|---|---|
| blend | 0.27783 | 0.109161 | 1101 |

**Runs on Kaggle** (GPU T4 x2, Internet on for `pip install`). Inputs and the saved-prediction chain are described in the [README](../README.md#reproducing).

In [3]:
import os, glob, time, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_data_dir():
    hits = glob.glob('/kaggle/input/**/Train.csv', recursive=True) + glob.glob('Train.csv')
    if not hits:
        raise FileNotFoundError('Train.csv not found - attach the dataset via "Add Input".')
    return os.path.dirname(os.path.abspath(hits[0]))

def gpu_available():
    try:
        return subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

DATA_DIR = find_data_dir()
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
USE_GPU = gpu_available()

# ---- knobs ----------------------------------------------------------------
FAST = os.environ.get('FAST') == '1'   # set True for a quick smoke test (1 seed, few trees)
N_FOLDS = 5
SEEDS = [0] if FAST else [0, 1, 2, 3, 4]   # repeated CV: more seeds = more stable OOF + smoother test preds
TUNE_CATBOOST = False            # optional Optuna search for CatBoost (GPU recommended)
TUNE_TRIALS = 30
PREVIOUS_BEST = 0.10926
# ---------------------------------------------------------------------------

print('data:', DATA_DIR, '| out:', OUT_DIR, '| GPU:', USE_GPU, '| FAST:', FAST)

data: /kaggle/input/datasets/jeffmogaka/datadef | out: /kaggle/working | GPU: True | FAST: False


## Data

In [4]:
train = pd.read_csv(f'{DATA_DIR}/Train.csv')
test = pd.read_csv(f'{DATA_DIR}/Test.csv')
y = (train['bank_account'] == 'Yes').astype(int).values
test_ids = test['uniqueid'] + ' x ' + test['country']

CATS = ['country', 'location_type', 'cellphone_access', 'gender_of_respondent',
        'relationship_with_head', 'marital_status', 'education_level', 'job_type']
NUMS = ['household_size', 'age_of_respondent']
# `year` is dropped: it maps 1:1 to country, so it adds nothing.

X = train[CATS + NUMS].reset_index(drop=True)
X_test = test[CATS + NUMS].reset_index(drop=True)

print(X.shape, X_test.shape, '| positive rate: %.4f' % y.mean())
print(train.groupby('country')['bank_account'].apply(lambda s: (s == 'Yes').mean()).round(3))

(23524, 10) (10086, 10) | positive rate: 0.1408
country
Kenya       0.251
Rwanda      0.115
Tanzania    0.092
Uganda      0.086
Name: bank_account, dtype: float64


## Cross-validation harness
Each model is a function `fit_predict(train_idx, valid_idx, seed) -> (valid_probs, test_probs)`. The harness runs it on the same folds for every seed, collecting OOF and averaged test predictions.

In [5]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

FOLDS = {s: list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y)) for s in SEEDS}
OOF, TEST, SCORES = {}, {}, {}

def error_rate(p, t=0.5):
    return np.mean((p > t).astype(int) != y)

def run_model(name, fit_predict):
    t0 = time.time()
    oof = np.zeros((len(SEEDS), len(y)))
    tst = np.zeros(len(X_test))
    for k, s in enumerate(SEEDS):
        for tr_idx, va_idx in FOLDS[s]:
            p_va, p_te = fit_predict(tr_idx, va_idx, s)
            oof[k, va_idx] = p_va
            tst += p_te / (len(SEEDS) * N_FOLDS)
    per_seed = [error_rate(o) for o in oof]
    OOF[name], TEST[name] = oof.mean(0), tst
    SCORES[name] = dict(cv_error=np.mean(per_seed), cv_error_std=np.std(per_seed),
                        logloss=np.mean([log_loss(y, o) for o in oof]), minutes=(time.time() - t0) / 60)
    print(f"{name:14s} CV error {np.mean(per_seed):.5f} ± {np.std(per_seed):.5f} | "
          f"logloss {SCORES[name]['logloss']:.5f} | {SCORES[name]['minutes']:.1f} min")

## Model 1 — CatBoost (GPU)

In [6]:
from catboost import CatBoostClassifier

CB_PARAMS = dict(iterations=300 if FAST else 1000, learning_rate=0.03, depth=6,
                 l2_leaf_reg=5, max_ctr_complexity=2, border_count=254)

if TUNE_CATBOOST:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        p = dict(iterations=1500,
                 learning_rate=trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
                 depth=trial.suggest_int('depth', 4, 8),
                 l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 30, log=True),
                 random_strength=trial.suggest_float('random_strength', 0.1, 5, log=True),
                 max_ctr_complexity=trial.suggest_int('max_ctr_complexity', 1, 3))
        losses = []
        for tr_idx, va_idx in FOLDS[SEEDS[0]]:
            m = make_catboost(p, SEEDS[0])
            m.fit(X.iloc[tr_idx], y[tr_idx], cat_features=CATS,
                  eval_set=(X.iloc[va_idx], y[va_idx]), early_stopping_rounds=100)
            losses.append(log_loss(y[va_idx], m.predict_proba(X.iloc[va_idx])[:, 1]))
        return np.mean(losses)   # tune on logloss: far less noisy than accuracy

def make_catboost(params, seed):
    extra = dict(task_type='GPU', devices='0') if USE_GPU else dict(thread_count=-1)
    return CatBoostClassifier(**params, **extra, random_seed=seed, verbose=0, allow_writing_files=False)

if TUNE_CATBOOST:
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=TUNE_TRIALS)
    print('best logloss %.5f' % study.best_value, study.best_params)
    CB_PARAMS.update(study.best_params)

def catboost_fp(tr_idx, va_idx, seed):
    m = make_catboost(CB_PARAMS, seed)
    m.fit(X.iloc[tr_idx], y[tr_idx], cat_features=CATS)
    return m.predict_proba(X.iloc[va_idx])[:, 1], m.predict_proba(X_test)[:, 1]

run_model('catboost', catboost_fp)

catboost       CV error 0.11156 ± 0.00036 | logloss 0.27926 | 15.6 min


## Model 2 — LightGBM (CPU, native categoricals)

In [7]:
import lightgbm as lgb

def as_category(df):
    df = df.copy()
    for c in CATS:
        df[c] = pd.Categorical(df[c], categories=sorted(set(X[c]) | set(X_test[c])))
    return df

Xc, Xc_test = as_category(X), as_category(X_test)

LGB_PARAMS = dict(n_estimators=150 if FAST else 600, learning_rate=0.02, num_leaves=31, min_child_samples=30,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, cat_smooth=20, reg_lambda=2,
                  verbose=-1, n_jobs=-1)

def lgb_fp(tr_idx, va_idx, seed):
    m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed)
    m.fit(Xc.iloc[tr_idx], y[tr_idx])
    return m.predict_proba(Xc.iloc[va_idx])[:, 1], m.predict_proba(Xc_test)[:, 1]

run_model('lightgbm', lgb_fp)

lightgbm       CV error 0.11216 ± 0.00027 | logloss 0.28114 | 0.6 min


## Model 3 — Logistic regression on pairwise crosses
A very different model family: every pair of (binned) features becomes a one-hot cross. Adds diversity to the blend.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

def crosses(df):
    d = df[CATS].copy()
    d['age_b'] = pd.cut(df['age_of_respondent'], [0, 20, 25, 30, 35, 40, 50, 60, 70, 200]).astype(str)
    d['hh_b'] = df['household_size'].clip(1, 12).astype(str)
    base = list(d.columns)
    for i in range(len(base)):
        for j in range(i + 1, len(base)):
            d[base[i] + '*' + base[j]] = d[base[i]] + '|' + d[base[j]]
    return d

Xx, Xx_test = crosses(X), crosses(X_test)

def lr_fp(tr_idx, va_idx, seed):
    enc = OneHotEncoder(handle_unknown='ignore', min_frequency=5)
    A = enc.fit_transform(Xx.iloc[tr_idx])
    m = LogisticRegression(C=0.05, max_iter=3000)
    m.fit(A, y[tr_idx])
    return m.predict_proba(enc.transform(Xx.iloc[va_idx]))[:, 1], m.predict_proba(enc.transform(Xx_test))[:, 1]

run_model('logreg_cross', lr_fp)

logreg_cross   CV error 0.11189 ± 0.00053 | logloss 0.28141 | 0.6 min


## Blend
Weights are fitted on OOF predictions by minimising log loss (non-negative, summing to 1). Then we check whether a cutoff other than 0.5 helps; it is only used if the gain is bigger than the seed-to-seed noise.

In [9]:
from scipy.optimize import minimize

names = list(OOF)
P = np.column_stack([OOF[n] for n in names])
T = np.column_stack([TEST[n] for n in names])

def blend_loss(w):
    w = np.abs(w) / np.abs(w).sum()
    return log_loss(y, np.clip(P @ w, 1e-6, 1 - 1e-6))

res = minimize(blend_loss, np.ones(len(names)) / len(names), method='Nelder-Mead',
               options=dict(maxiter=4000, xatol=1e-5, fatol=1e-7))
W = np.abs(res.x) / np.abs(res.x).sum()
oof_blend, test_blend = P @ W, T @ W

print('blend weights:', dict(zip(names, W.round(3))))
print('blend  CV error %.5f | logloss %.5f' % (error_rate(oof_blend), log_loss(y, oof_blend)))

grid = np.round(np.arange(0.40, 0.601, 0.01), 2)
errs = np.array([error_rate(oof_blend, t) for t in grid])
best_t = grid[errs.argmin()]
# Require at least ~0.001 (about 24 training rows) on top of the seed-to-seed noise;
# smaller 'gains' from moving the cutoff are just fitting the OOF noise.
noise = max(np.mean([SCORES[n]['cv_error_std'] for n in names]), 0.001)
THRESH = best_t if errs.min() < error_rate(oof_blend) - noise else 0.5
print(f'best cutoff {best_t} (err {errs.min():.5f}) | noise ±{noise:.5f} -> using cutoff {THRESH}')

blend weights: {'catboost': np.float64(0.615), 'lightgbm': np.float64(0.207), 'logreg_cross': np.float64(0.179)}
blend  CV error 0.11172 | logloss 0.27783
best cutoff 0.54 (err 0.11104) | noise ±0.00100 -> using cutoff 0.5


## Leaderboard of models

In [10]:
board = pd.DataFrame(SCORES).T
board.loc['BLEND'] = [error_rate(oof_blend, THRESH), np.nan, log_loss(y, oof_blend), board['minutes'].sum()]
board['vs_previous_best'] = board['cv_error'] - PREVIOUS_BEST
display(board.sort_values('cv_error').round(5))

,cv_error,cv_error_std,logloss,minutes,vs_previous_best
catboost,0.11156,0.00036,0.27926,15.64225,0.00230
BLEND,0.11172,NaN,0.27783,16.86550,0.00246
logreg_cross,0.11189,0.00053,0.28141,0.64486,0.00263
lightgbm,0.11216,0.00027,0.28114,0.57838,0.00290


## Write submissions
`submission_blend.csv` is the main one. The best single model is also saved, for comparison on the public leaderboard.

In [11]:
def save(name, probs, t=0.5):
    sub = pd.DataFrame({'unique_id': test_ids, 'bank_account': (probs > t).astype(int)})
    path = f'{OUT_DIR}/submission_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'{path}: {sub.bank_account.sum()} ones / {len(sub)} rows')

save('blend', test_blend, THRESH)
best_single = min(names, key=lambda n: SCORES[n]['cv_error'])
save(f'single_{best_single}', TEST[best_single])

# Keep the raw probabilities too, for later blending with other runs.
pd.DataFrame({'unique_id': test_ids, **{n: TEST[n] for n in names}, 'blend': test_blend}) \
  .to_csv(f'{OUT_DIR}/test_probabilities.csv', index=False)

/kaggle/working/submission_blend.csv: 751 ones / 10086 rows
/kaggle/working/submission_single_catboost.csv: 742 ones / 10086 rows
